# Notebook 18 — Activity-Cliff Contrastive Representations

**Question:** Does an explicitly cliff-aware molecular representation help correct errors of the already strong CheMeleon + screening ensemble?

Mines **within-isoform, training-fold-only** structurally similar pairs with small and large potency differences, learns a contrastive 32-dimensional projection of Morgan bits, and tests whether ridge regression on those features improves ensemble residuals. Controls include raw fingerprints, an untrained random projection, and a projection trained on shuffled activity pair labels.

This is a controlled, CPU-accessible **contrastive projection experiment**, **not** an end-to-end fine-tune of CheMeleon. Pair scarcity and fallback behavior are explicitly logged. Its goal is to measure whether activity-cliff-aware representations add incremental information, before investing GPU time in graph-model fine-tuning.

## 0. Install packages
Run once in Colab. Python ≥3.10 recommended. A GPU is optional for the neural or contrastive experiment; the pretrained CheMeleon model is **not retrained**.

In [ ]:
%pip -q install "numpy>=1.26" "pandas>=2.0" "rdkit>=2023.09" "scikit-learn>=1.4" "scipy>=1.11" "matplotlib>=3.8" "torch>=2.1"

## 1. Configure and upload Notebook 11 results
Download `cyp_notebook_11_results.zip` from the **real five-fold Notebook 11 run**, upload it to Colab, and set its path. Use a Google Drive `PERSISTENT_ROOT` to make interrupted executions restartable. Run `FAST_MODE=True` first; use `False` to produce research-scale tables.

In [ ]:
from pathlib import Path
import os, random
SEED = 2026
FAST_MODE = True  # first run uses outer fold 0; change to False for all five
FOLDS_TO_RUN = [0] if FAST_MODE else [0,1,2,3,4]
FP_BITS = 256  # choose 512 for final full run if CPU budget permits
NB11_RESULTS_ZIP = '/content/cyp_notebook_11_results.zip' if Path('/content').exists() else '/mnt/data/cyp_notebook_11_results.zip'
PERSISTENT_ROOT = None  # e.g. '/content/drive/MyDrive/CYP/PotencyRegime/18'
EXPERIMENT_TAG = 'regime_v1'  # increment after code/method changes to avoid stale caches
OUTPUT = (Path(PERSISTENT_ROOT) if PERSISTENT_ROOT else Path('.')) / 'cyp_notebook_18_results'
OUTPUT.mkdir(parents=True,exist_ok=True)
assert Path(NB11_RESULTS_ZIP).is_file(), 'Upload the REAL full five-fold Notebook 11 results ZIP and edit NB11_RESULTS_ZIP'
print('Notebook 18', 'FAST' if FAST_MODE else 'FIVE-FOLD', 'cache directory:', OUTPUT)


## 2. Load and audit the saved first-stage OOF predictions
The following standalone utilities parse the archive, confirm the baseline is the fixed 50:50 CheMeleon/screening blend, reconstruct inner-training CI bounds from the corresponding training-compound records, and **assert no outer held-out molecule or scaffold appears in the fitting data**.

In [ ]:
"""Common data handling and leakage-safe utilities embedded into standalone CYP Colab notebooks.
Input is the complete five-fold Notebook 11 results archive. No outer-fold targets enter fitting.
"""
import io, json, zipfile, hashlib, warnings, random, os, sys, math, time
from pathlib import Path
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs, rdBase
from rdkit.Chem import Descriptors, rdFingerprintGenerator, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.metrics import roc_auc_score, brier_score_loss, mean_absolute_error
from scipy.optimize import minimize
ISOS=['CYP1A2','CYP2C9','CYP2D6','CYP3A4']
BASE_NAME='CheMeleon_finetuned + ScreenStack_LGBM'
CHEM_NAME='CheMeleon_finetuned'
SCREEN_NAME='ScreenStack_LGBM'
DESCRIPTOR_FNS=[('MolWt',Descriptors.MolWt),('MolLogP',Descriptors.MolLogP),('TPSA',Descriptors.TPSA),('NumHDonors',Descriptors.NumHDonors),('NumHAcceptors',Descriptors.NumHAcceptors),('NumRotatableBonds',Descriptors.NumRotatableBonds),('RingCount',Descriptors.RingCount),('FractionCSP3',Descriptors.FractionCSP3),('HeavyAtomCount',Descriptors.HeavyAtomCount),('NumAromaticRings',Descriptors.NumAromaticRings),('NumAliphaticRings',Descriptors.NumAliphaticRings),('NumHeteroatoms',Descriptors.NumHeteroatoms)]

def zread(z,leaf):
    paths=[n for n in z.namelist() if n == leaf or n.endswith('/'+leaf)]
    if len(paths)!=1:raise ValueError(f'{leaf}: expected exactly one archive member, found {paths}')
    return z.read(paths[0])

def load_archive(path,allow_incomplete=False):
    with zipfile.ZipFile(path) as z:
        outer=pd.read_csv(io.BytesIO(zread(z,'nested_outer_predictions.csv')))
        inner=pd.read_csv(io.BytesIO(zread(z,'inner_oof_predictions_for_selection.csv')))
        manifest=pd.read_csv(io.BytesIO(zread(z,'fold_manifest.csv')))
        metadata=json.loads(zread(z,'run_metadata.json'))
    if not allow_incomplete:
        assert metadata.get('scientifically_valid_full_model_run') and metadata.get('complete_five_fold_oof'), 'Need full scientifically valid Notebook 11 output, not proxy/FAST'
    assert {'canonical_smiles','scaffold_group','fold'}<=set(manifest.columns)
    assert manifest.canonical_smiles.is_unique and manifest.scaffold_group.notna().all()
    assert manifest.groupby('scaffold_group').fold.nunique().max()==1
    o=outer[outer.subset.isin([BASE_NAME,CHEM_NAME,SCREEN_NAME])].copy()
    assert set(o.subset.unique())=={BASE_NAME,CHEM_NAME,SCREEN_NAME},'Missing component predictions in Notebook 11 archive'
    assert o[['canonical_smiles','isoform','fold','subset']].duplicated().sum()==0
    names=['y_true','ci_low','ci_high','scaffold_group']
    baseline=o[o.subset==BASE_NAME][['canonical_smiles','isoform','fold','y_true','y_pred','ci_low','ci_high','scaffold_group']].copy().rename(columns={'y_pred':'base'})
    c=o[o.subset==CHEM_NAME][['canonical_smiles','isoform','fold','y_pred']].rename(columns={'y_pred':'chemeleon'})
    s=o[o.subset==SCREEN_NAME][['canonical_smiles','isoform','fold','y_pred']].rename(columns={'y_pred':'screenstack'})
    outer_data=baseline.merge(c,on=['canonical_smiles','isoform','fold'],validate='one_to_one').merge(s,on=['canonical_smiles','isoform','fold'],validate='one_to_one')
    # For this *fixed* candidate baseline use explicitly 50:50 rather than Notebook11's selected pair label.
    assert np.allclose(outer_data.base,0.5*(outer_data.chemeleon+outer_data.screenstack),atol=1e-6)
    assert outer_data.isna().sum().get('y_true',0)==0
    assert set(outer_data.isoform)==set(ISOS)
    inner=inner.copy()
    assert not inner.duplicated(['fold','canonical_smiles','isoform']).any()
    inner['base']=0.5*(inner[CHEM_NAME]+inner[SCREEN_NAME])
    inner['chemeleon']=inner[CHEM_NAME]
    inner['screenstack']=inner[SCREEN_NAME]
    # The outer-prediction file contains CI for every compound *in its own outer fold*.
    # Using that label metadata for molecules INSIDE the current training partition is valid.
    reference=outer_data[['canonical_smiles','isoform','y_true','ci_low','ci_high']]
    inner=inner.merge(reference,on=['canonical_smiles','isoform'],suffixes=('','_outer'),validate='many_to_one')
    assert np.allclose(inner.y_true,inner.y_true_outer,atol=1e-6)
    inner=inner.drop(columns=['y_true_outer'])
    inner=inner.merge(manifest[['canonical_smiles','scaffold_group','fold']].rename(columns={'fold':'true_outer_fold'}),on='canonical_smiles',validate='many_to_one')
    assert (inner.fold!=inner.true_outer_fold).all(),'Inner meta-training contains an outer held-out compound'
    assert (outer_data.groupby('canonical_smiles').fold.nunique()==1).all()
    for fold in sorted(outer_data.fold.unique()):
        train=inner[inner.fold==fold]; held=outer_data[outer_data.fold==fold]
        assert not set(train.canonical_smiles)&set(held.canonical_smiles)
        assert not set(train.scaffold_group)&set(held.scaffold_group)
    assert np.isfinite(outer_data[['base','chemeleon','screenstack','y_true']].to_numpy(float)).all()
    assert np.isfinite(inner[['base','chemeleon','screenstack','y_true']].to_numpy(float)).all()
    return outer_data,inner,manifest,metadata

def fingerprint_features(smiles_list,bits=512):
    gen=rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=bits)
    X=np.empty((len(smiles_list),bits+len(DESCRIPTOR_FNS)),dtype=np.float32)
    fps=[]
    for i,s in enumerate(smiles_list):
        m=Chem.MolFromSmiles(s)
        if m is None:raise ValueError(f'Invalid SMILES: {s!r}')
        fp=gen.GetFingerprint(m);fps.append(fp)
        arr=np.zeros(bits,dtype=np.int8);DataStructs.ConvertToNumpyArray(fp,arr)
        X[i,:bits]=arr
        for j,(_,fn) in enumerate(DESCRIPTOR_FNS):
            try:v=fn(m);X[i,bits+j]=float(v) if np.isfinite(v) else 0.
            except Exception:X[i,bits+j]=0.
    return X,fps

def feature_table(outer,inner,bits=512):
    all_smiles=sorted(set(outer.canonical_smiles)|set(inner.canonical_smiles))
    arr,fps=fingerprint_features(all_smiles,bits)
    index={s:k for k,s in enumerate(all_smiles)}
    return {'x':arr,'fps':fps,'smiles':all_smiles,'index':index,'bits':bits}

def make_x(frame,features,include_model=True):
    ids=np.fromiter((features['index'][s] for s in frame.canonical_smiles),dtype=int,count=len(frame))
    desc=features['x'][ids]
    if include_model:
        additional=np.stack([frame.base.to_numpy(float),frame.chemeleon.to_numpy(float),frame.screenstack.to_numpy(float),
                             (frame.chemeleon-frame.screenstack).to_numpy(float),
                             np.asarray([ISOS.index(k) for k in frame.isoform],float)/3],axis=1).astype(np.float32)
        return np.column_stack([desc,additional])
    return desc

def metric_stats(frame,pred,original=None):
    local=frame.copy();local['p']=np.asarray(pred,float)
    local['err']=(local.y_true-local.p).abs()
    lo=local.ci_low.to_numpy(float);hi=local.ci_high.to_numpy(float);p=local.p.to_numpy(float)
    valid=np.isfinite(lo)&np.isfinite(hi)&(lo<=hi)
    dist=np.maximum(lo-p,0)+np.maximum(p-hi,0)
    local['excess']=np.where(valid,dist,np.nan)
    out={'macro_mae':float(local.groupby('isoform').err.mean().reindex(ISOS).mean()),
         'macro_interval_excess':float(local.groupby('isoform').excess.mean().reindex(ISOS).mean()),
         'n':len(local)}
    if original is not None:
        out['mae_improvement']=float(metric_stats(frame,original)['macro_mae']-out['macro_mae'])
    return out

def safe_auc(y,p):
    return float(roc_auc_score(y,p)) if len(set(np.asarray(y,dtype=int)))==2 else np.nan

def sample_bootstrap(all_preds,baseline_col='base',B=300,seed=2026):
    p=all_preds.copy(); p['benefit']=(p.y_true-p[baseline_col]).abs()-(p.y_true-p.y_pred).abs()
    ss=p.groupby(['scaffold_group','isoform']).benefit.agg(['sum','count']).reset_index()
    sums=ss.pivot(index='scaffold_group',columns='isoform',values='sum').reindex(columns=ISOS).fillna(0).to_numpy()
    cnt=ss.pivot(index='scaffold_group',columns='isoform',values='count').reindex(columns=ISOS).fillna(0).to_numpy()
    rng=np.random.default_rng(seed);v=[]
    for _ in range(B):
        ind=rng.integers(0,len(sums),len(sums));nr=sums[ind].sum(0);dn=cnt[ind].sum(0)
        if (dn>0).all():v.append(float(np.mean(nr/dn)))
    return {'gain':float(p.groupby('isoform').benefit.mean().reindex(ISOS).mean()),
            'low':float(np.quantile(v,.025)) if len(v) else np.nan,
            'high':float(np.quantile(v,.975)) if len(v) else np.nan,'reps':len(v)}

def wrap_results(output,archive_filename,files):
    import zipfile
    with zipfile.ZipFile(archive_filename,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for fn in files:
            path=Path(output)/fn
            if path.exists():z.write(path,arcname='cyp_regime_results/'+fn)
    print('Wrote',archive_filename)

def guarded_folds(train,held):
    assert not set(train.canonical_smiles)&set(held.canonical_smiles)
    assert not set(train.scaffold_group)&set(held.scaffold_group)

def save_csv(df,path):
    # Normalize floating precision across fresh and cached executions.
    df.to_csv(path,index=False,float_format='%.10g')

def get_root():
    return Path(PERSISTENT_ROOT) if PERSISTENT_ROOT else Path('.')


In [ ]:
OUTER, INNER, MANIFEST, META11 = load_archive(NB11_RESULTS_ZIP)
FEATURES = feature_table(OUTER, INNER, bits=FP_BITS)
print("Outer observations:", len(OUTER), "Inner meta observations:", len(INNER), "Features:", FEATURES["x"].shape)
print("Source scientifically valid:", META11.get("scientifically_valid_full_model_run"))
assert OUTER.groupby("canonical_smiles").fold.nunique().max()==1

## 3. Fold-safe modeling and ablation
**No outer-fold potency labels are used for fitting, selecting a correction, calibrating a gate, or mining training pairs.** The methods cache each completed fold. Expected outputs include a chosen-method audit, per-CYP performance, and low/high-potency diagnostics.

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

torch.set_num_threads(min(4,max(1,os.cpu_count() or 1)))
PAIR_SIM=.55
MIN_CLIFF_PAIRS=8
MAX_PAIRS=900
EPOCHS=15 if FAST_MODE else 55
LAMBDA_GRID=[0.,.15,.35,.6]
METHODS=['RawFingerprintResidual','RandomProjectionResidual','ContrastiveResidual','ShuffledPairResidual']

class PairEncoder(nn.Module):
    def __init__(self,n_in,seed=SEED):
        super().__init__();torch.manual_seed(seed)
        self.net=nn.Sequential(nn.Linear(n_in,80),nn.ReLU(),nn.Linear(80,32))
    def forward(self,x):return F.normalize(self.net(x),p=2,dim=1)

def chemical_pairs(train,kind,seed):
    # Pair mining uses only the current fitting partition's labels and molecular structures.
    rng=np.random.default_rng(seed)
    fps=[FEATURES['fps'][FEATURES['index'][s]] for s in train.canonical_smiles]
    y=train.y_true.to_numpy(float)
    if kind=='shuffled':y=rng.permutation(y)
    positives=[];negative=[]
    for threshold in [PAIR_SIM,.45,.35]:
        positives=[];negative=[]
        for i in range(len(fps)-1):
            sims=np.asarray(DataStructs.BulkTanimotoSimilarity(fps[i],fps[i+1:]),dtype=np.float32)
            eligible=np.flatnonzero(sims>=threshold)+i+1
            if not len(eligible):continue
            diffs=np.abs(y[i]-y[eligible])
            positives.extend((i,int(j)) for j in eligible[diffs<=.35])
            negative.extend((i,int(j)) for j in eligible[diffs>=1.0])
        if len(negative)>=MIN_CLIFF_PAIRS and len(positives)>=MIN_CLIFF_PAIRS:break
    # cap using a stable random seed so every experiment is bounded and restartable
    if len(positives)>MAX_PAIRS:
        ix=rng.choice(len(positives),size=MAX_PAIRS,replace=False);positives=[positives[t] for t in ix]
    if len(negative)>MAX_PAIRS:
        ix=rng.choice(len(negative),size=MAX_PAIRS,replace=False);negative=[negative[t] for t in ix]
    return positives,negative,threshold

def learn_projection(train,features,seed,mode):
    x=make_x(train,features,include_model=False)
    scale=StandardScaler().fit(x);x=np.clip(scale.transform(x),-8,8).astype(np.float32)
    net=PairEncoder(x.shape[1],seed)
    audit={'n_pos':0,'n_cliff':0,'similarity_min':np.nan,'trained':False,'epochs':0}
    if mode=='random':return net,scale,audit
    pos,neg,cut=chemical_pairs(train,'shuffled' if mode=='shuffled' else 'true',seed)
    audit.update(n_pos=len(pos),n_cliff=len(neg),similarity_min=cut)
    if min(len(pos),len(neg))<MIN_CLIFF_PAIRS:return None,scale,audit
    p=torch.tensor(np.asarray(pos),dtype=torch.int64);n=torch.tensor(np.asarray(neg),dtype=torch.int64)
    xx=torch.tensor(x);opt=torch.optim.AdamW(net.parameters(),lr=.001,weight_decay=.01)
    for epoch in range(EPOCHS):
        opt.zero_grad();z=net(xx)
        posdist=((z[p[:,0]]-z[p[:,1]])**2).sum(1)
        negdist=torch.sqrt(((z[n[:,0]]-z[n[:,1]])**2).sum(1)+1e-8)
        contrast=posdist.mean()+torch.relu(1.15-negdist).square().mean()
        # Prevent representation collapse, preserve modest input variance.
        spread=torch.relu(.08-z.std(dim=0)).mean()
        loss=contrast+.1*spread;loss.backward();torch.nn.utils.clip_grad_norm_(net.parameters(),2.0);opt.step()
    audit.update(trained=True,epochs=epoch+1)
    net.eval();return net,scale,audit

def projected(net,scale,frame,features):
    x=np.clip(scale.transform(make_x(frame,features,include_model=False)),-8,8).astype(np.float32)
    with torch.no_grad():return net(torch.tensor(x)).numpy().astype(float)

def predict_correction(train,held,method,seed):
    a=train.base.to_numpy(float);b=held.base.to_numpy(float)
    if method=='RawFingerprintResidual':
        xtr=make_x(train,FEATURES,include_model=False);xte=make_x(held,FEATURES,include_model=False)
        imp=StandardScaler().fit(xtr);a1=np.clip(imp.transform(xtr),-8,8);b1=np.clip(imp.transform(xte),-8,8)
        audit={'n_pos':0,'n_cliff':0,'similarity_min':np.nan,'trained':True,'epochs':0}
    else:
        mode={'RandomProjectionResidual':'random','ContrastiveResidual':'contrastive','ShuffledPairResidual':'shuffled'}[method]
        model,scale,audit=learn_projection(train,FEATURES,seed,mode)
        if model is None:return np.zeros(len(held),float),audit
        a1=projected(model,scale,train,FEATURES);b1=projected(model,scale,held,FEATURES)
    reg=Ridge(alpha=50. if method=='RawFingerprintResidual' else 10.)
    reg.fit(a1,train.y_true.to_numpy(float)-a)
    return np.clip(reg.predict(b1),-1.5,1.5),audit

def selection_cv(meta,seed):
    oof={k:np.full(len(meta),np.nan) for k in METHODS}
    for f,(tr,va) in enumerate(GroupKFold(n_splits=2).split(meta,groups=meta.scaffold_group)):
        train=meta.iloc[tr];held=meta.iloc[va];guarded_folds(train,held)
        for j,method in enumerate(METHODS):
            correction,_=predict_correction(train,held,method,seed+f*100+j)
            oof[method][va]=correction
    selected={};base=meta.base.to_numpy(float);y=meta.y_true.to_numpy(float)
    for method in METHODS:
        scored=[(float(np.mean(abs(y-(base+v*oof[method])))),float(v)) for v in LAMBDA_GRID]
        selected[method]=min(scored,key=lambda t:(t[0],t[1]))
    return selected

sig=hashlib.sha256(json.dumps({'tag':EXPERIMENT_TAG,'nb11':hashlib.sha256(Path(NB11_RESULTS_ZIP).read_bytes()).hexdigest()[:12],
    'bits':FP_BITS,'fast':FAST_MODE,'epoch':EPOCHS,'minpair':MIN_CLIFF_PAIRS,'sim':PAIR_SIM,'seed':SEED},sort_keys=True).encode()).hexdigest()[:16]
CACHE=OUTPUT/'cache18'/sig;CACHE.mkdir(parents=True,exist_ok=True)
allpred=[];allsel=[];allpairs=[];allaudit=[]
for fold in FOLDS_TO_RUN:
    cp=CACHE/f'{fold}_pred.csv';sp=CACHE/f'{fold}_select.csv';pp=CACHE/f'{fold}_pairs.csv';ap=CACHE/f'{fold}_audit.csv'
    if all(p.exists() for p in [cp,sp,pp,ap]):
        allpred.append(pd.read_csv(cp));allsel.append(pd.read_csv(sp));allpairs.append(pd.read_csv(pp));allaudit.append(pd.read_csv(ap));print('Cache hit',fold);continue
    train_all=INNER[INNER.fold==fold];held_all=OUTER[OUTER.fold==fold];guarded_folds(train_all,held_all)
    records=[];sels=[];pairs=[];audits=[]
    for j,iso in enumerate(ISOS):
        train=train_all[train_all.isoform==iso].reset_index(drop=True)
        held=held_all[held_all.isoform==iso].reset_index(drop=True)
        guarded_folds(train,held)
        choices=selection_cv(train,SEED+fold*1000+j*100)
        correction={};diag={}
        for method_idx,method in enumerate(METHODS):
            corr,aud=predict_correction(train,held,method,SEED+fold*1000+j*100+method_idx+900)
            correction[method]=corr
            diag[method]=aud
            pairs.append({'fold':fold,'isoform':iso,'method':method,**aud})
        champ=min(choices,key=lambda k:(choices[k][0],METHODS.index(k)))
        for method,(score,weight) in choices.items():sels.append({'fold':fold,'isoform':iso,'method':method,'inner_cv_mae':score,'selected_strength':weight,'selected':method==champ})
        for method in ['Baseline']+METHODS+['NestedSelected']:
            if method=='Baseline':pred=held.base.to_numpy(float);src='Baseline';lam=0
            elif method=='NestedSelected':src=champ;lam=choices[champ][1];pred=held.base.to_numpy(float)+lam*correction[champ]
            else:src=method;lam=choices[method][1];pred=held.base.to_numpy(float)+lam*correction[method]
            for i,row in held.iterrows():
                records.append({'fold':fold,'isoform':iso,'method':method,'canonical_smiles':row.canonical_smiles,
                     'scaffold_group':row.scaffold_group,'y_true':row.y_true,'ci_low':row.ci_low,'ci_high':row.ci_high,
                     'base':row.base,'y_pred':float(pred[i]),'correction_strength':lam})
        audits.append({'fold':fold,'isoform':iso,'n_train':len(train),'n_held':len(held),
           'scaffold_overlap':len(set(train.scaffold_group)&set(held.scaffold_group)),
           'num_mined_cliffs':diag['ContrastiveResidual']['n_cliff'],
           'num_mined_neighbors':diag['ContrastiveResidual']['n_pos']})
    for obj,path in [(records,cp),(sels,sp),(pairs,pp),(audits,ap)]:save_csv(pd.DataFrame(obj),path)
    allpred.append(pd.read_csv(cp));allsel.append(pd.read_csv(sp));allpairs.append(pd.read_csv(pp));allaudit.append(pd.read_csv(ap))
    print('Finished fold',fold)

outer=pd.concat(allpred,ignore_index=True);chosen=pd.concat(allsel,ignore_index=True);pairs=pd.concat(allpairs,ignore_index=True);audit=pd.concat(allaudit,ignore_index=True)
assert (audit.scaffold_overlap==0).all()
assert not outer.duplicated(['method','canonical_smiles','isoform']).any()
summary=[];per=[];extremes=[]
for method,g in outer.groupby('method'):
    summary.append({'method':method,**metric_stats(g,g.y_pred)})
    for iso,p in g.groupby('isoform'):
        per.append({'method':method,'isoform':iso,'n':len(p),'mae':float(np.mean(abs(p.y_true-p.y_pred)))})
        low,high=np.quantile(p.y_true,[.1,.9]);
        for tier,rows in [('low',p[p.y_true<=low]),('high',p[p.y_true>=high])]:
            extremes.append({'method':method,'isoform':iso,'tier':tier,'n':len(rows),
               'mae':float(np.mean(abs(rows.y_true-rows.y_pred)))})
summary=pd.DataFrame(summary).sort_values('macro_mae')
for name,obj in [('outer_predictions.csv',outer),('macro_summary.csv',summary),('isoform_summary.csv',pd.DataFrame(per)),
                 ('potency_extremes.csv',pd.DataFrame(extremes)),('nested_strengths.csv',chosen),
                 ('activity_cliff_pair_audit.csv',pairs),('leakage_audit.csv',audit)]:save_csv(obj,OUTPUT/name)
boot=sample_bootstrap(outer[outer.method=='NestedSelected'],B=150 if FAST_MODE else 1000)
save_csv(pd.DataFrame([boot]),OUTPUT/'paired_scaffold_bootstrap.csv')
(OUTPUT/'run_metadata.json').write_text(json.dumps({'experiment':'Activity-cliff contrastive projection of Morgan fingerprints',
 'input':'Notebook11 real five-fold inner OOF and outer OOF',
 'caveats':['This trains a small contrastive projection on Morgan fingerprints, NOT full CheMeleon message passing weights.',
            'Only training-partition activity labels enter pair mining. Sparse cliffs trigger a no-correction fallback.',
            'CV selects strengths; first-stage model predictions for inner meta rows were previously cross-fitted.'],
 'settings':{'fingerprint_bits':FP_BITS,'pair_similarity':PAIR_SIM,'min_cliffs':MIN_CLIFF_PAIRS,'epochs':EPOCHS,'fast':FAST_MODE},'folds':FOLDS_TO_RUN},indent=2))
wrap_results(OUTPUT,'cyp_notebook_18_results.zip',['outer_predictions.csv','macro_summary.csv','isoform_summary.csv','potency_extremes.csv','nested_strengths.csv','activity_cliff_pair_audit.csv','leakage_audit.csv','paired_scaffold_bootstrap.csv','run_metadata.json'])
print(summary.to_string(index=False));print('Activity cliff training pairs:');print(pairs.groupby(['isoform','method'])[['n_pos','n_cliff']].sum().to_string())


## 4. Visual comparison and research interpretation
Inspect the outer-fold tables before drawing conclusions. Improvements are exploratory unless repeated across seeds and split strategies. In particular, compare potent-decile error, interval-aware performance, gate/contrastive diagnostics, and robustness to removing each modeling component.

In [ ]:
import matplotlib.pyplot as plt
try:
    chart_df = summary.sort_values('macro_mae' if 'macro_mae' in summary else 'macro_st_rae')
    key = 'macro_st_rae' if 'macro_st_rae' in summary else 'macro_mae'
    fig,ax=plt.subplots(figsize=(9,4))
    ax.barh(chart_df.method[::-1],chart_df[key][::-1])
    ax.set_xlabel(key+' (lower is better)');ax.set_title('Nested outer-fold comparison')
    plt.tight_layout();plt.show()
except Exception as e:
    print('Optional figure unavailable:',type(e).__name__,str(e))


## 5. Reproducibility checklist
- [ ] Real Notebook 11 archive, no proxy checkpoint
- [ ] `FAST_MODE=False`, outer folds 0–4
- [ ] No held-out scaffold overlaps in audits
- [ ] No selecting thresholds/strengths from outer-fold outcomes
- [ ] Compare against the **fixed** 50:50 CheMeleon + screening baseline on exactly matched molecules
- [ ] Publish a negative result if the specialists fail to improve the strong baseline
- [ ] Record dataset versions, code revisions, and multiple seeds for further research